In [ ]:
import os
os.chdir('/mmsegmentation')


from pathlib import Path
from typing import Dict, Any, Optional

import torch
import pandas as pd
from mmseg.apis import init_model
from mmseg.utils import register_all_modules



# =========================================================
# ADJUST THESE PATHS
# =========================================================
MODELS_TO_CHECK = {
    'multitask': {
        'config': '/mmsegmentation/zmax_configs/for_test_hasta_26_3/multitask_test_sin_trigger_clean.py',
        'checkpoint': None,  # optional
    },
    'seg_only': {
        'config': '/mmsegmentation/zmax_configs/for_test_hasta_26_3/bisenet_seg_only_test_clean.py',
        'checkpoint': None,  # optional
    },
    'cls_only': {
        'config': '/mmsegmentation/zmax_configs/for_test_hasta_26_3/bisenet_cls_only_test_clean.py',
        'checkpoint': None,  # optional
    },
}

DEVICE = 'cpu'  # GPU is not required to count parameters



In [ ]:

# =========================
# IMPORTS
# =========================

from pathlib import Path
from collections import OrderedDict
import os
import re
import warnings
import math

import pandas as pd
import torch
from IPython.display import display

from mmseg.apis import init_model
from mmseg.utils import register_all_modules

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 100)


In [ ]:

# =========================
# HELPER FUNCTIONS
# =========================

def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    non_trainable = total - trainable
    return total, trainable, non_trainable

def fmt_m(n):
    return f'{n:,} ({n/1e6:.3f} M)'

def safe_build_model(config_path, checkpoint=None, device='cpu'):
    register_all_modules(init_default_scope=False)
    model = init_model(config_path, checkpoint=checkpoint, device=device)
    return model

def get_submodule_by_path(root_module, path):
    if path in (None, '', '.'):
        return root_module
    cur = root_module
    for part in path.split('.'):
        if not hasattr(cur, part):
            return None
        cur = getattr(cur, part)
    return cur

def child_breakdown(module, model_name, parent_path):
    rows = []
    for child_name, child_module in module.named_children():
        full_path = f'{parent_path}.{child_name}' if parent_path else child_name
        total, trainable, non_trainable = count_params(child_module)
        rows.append({
            'model_name': model_name,
            'parent_path': parent_path if parent_path else '<root>',
            'module_name': child_name,
            'full_path': full_path,
            'module_type': child_module.__class__.__name__,
            'total_params': total,
            'trainable_params': trainable,
            'non_trainable_params': non_trainable,
            'total_params_M': total / 1e6,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('total_params', ascending=False).reset_index(drop=True)
    return df

def all_named_modules_breakdown(module, model_name, root_label='model'):
    rows = []
    for name, submodule in module.named_modules():
        path = root_label if name == '' else f'{root_label}.{name}'
        total, trainable, non_trainable = count_params(submodule)
        rows.append({
            'model_name': model_name,
            'full_path': path,
            'relative_path': name if name != '' else '<root>',
            'module_type': submodule.__class__.__name__,
            'total_params': total,
            'trainable_params': trainable,
            'non_trainable_params': non_trainable,
            'n_children_direct': len(list(submodule.named_children())),
            'total_params_M': total / 1e6,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['total_params', 'full_path'], ascending=[False, True]).reset_index(drop=True)
    return df

def find_interesting_modules(named_modules_dict):
    # Look for typical names, without assuming they always exist
    exact_candidates = [
        'backbone',
        'decode_head',
        'cls_head',
        'backbone.spatial_path',
        'backbone.context_path',
        'backbone.ffm',
        'backbone.feature_fusion_module',
        'backbone.context_path.arms',
        'backbone.context_path.attention_refinement_modules',
        'backbone.context_path.arm16',
        'backbone.context_path.arm32',
        'backbone.context_path.conv_head16',
        'backbone.context_path.conv_head32',
        'backbone.context_path.gap_conv',
    ]
    rows = []
    for path in exact_candidates:
        mod = named_modules_dict.get(path, None)
        if mod is not None:
            total, trainable, non_trainable = count_params(mod)
            rows.append({
                'full_path': path,
                'module_type': mod.__class__.__name__,
                'total_params': total,
                'trainable_params': trainable,
                'non_trainable_params': non_trainable,
                'total_params_M': total / 1e6,
                'match_type': 'exact_candidate'
            })
    return pd.DataFrame(rows)

def find_attention_like_modules(named_modules_dict):
    rows = []
    seen = set()
    for path, mod in named_modules_dict.items():
        rel = path.lower()
        typ = mod.__class__.__name__.lower()
        if (
            'attention' in rel or
            'attention' in typ or
            rel.endswith('.arms') or
            '.arms.' in rel or
            rel.endswith('.arm16') or
            rel.endswith('.arm32') or
            'refinement' in rel or
            'refinement' in typ
        ):
            if path in seen:
                continue
            seen.add(path)
            total, trainable, non_trainable = count_params(mod)
            rows.append({
                'full_path': path,
                'module_type': mod.__class__.__name__,
                'total_params': total,
                'trainable_params': trainable,
                'non_trainable_params': non_trainable,
                'total_params_M': total / 1e6,
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('total_params', ascending=False).reset_index(drop=True)
    return df

def preview_df(title, df, n=20):
    print(f'\n===== {title} =====')
    if df is None or df.empty:
        print('(vacío)')
    else:
        display(df.head(n))


In [ ]:

# =========================
# DETAILED COUNT
# =========================

summary_rows = []
root_children_tables = {}
backbone_children_tables = {}
context_children_tables = {}
spatial_children_tables = {}
fusion_children_tables = {}
attention_tables = {}
interesting_tables = {}
all_modules_tables = {}

models_cache = {}

for model_name, info in MODELS_TO_CHECK.items():
    print(f'\n### Construyendo: {model_name}')
    cfg = info['config']
    ckpt = info.get('checkpoint', None)

    model = safe_build_model(cfg, checkpoint=ckpt, device='cpu')
    models_cache[model_name] = model

    total, trainable, non_trainable = count_params(model)

    summary_rows.append({
        'model_name': model_name,
        'config': cfg,
        'model_type': model.__class__.__name__,
        'total_params': total,
        'trainable_params': trainable,
        'non_trainable_params': non_trainable,
        'total_params_M': total / 1e6,
        'trainable_params_M': trainable / 1e6,
        'non_trainable_params_M': non_trainable / 1e6,
    })

    # Direct children of the model
    root_children_tables[model_name] = child_breakdown(model, model_name, '')

    # Direct children of the backbone
    backbone = getattr(model, 'backbone', None)
    if backbone is not None:
        backbone_children_tables[model_name] = child_breakdown(backbone, model_name, 'backbone')
    else:
        backbone_children_tables[model_name] = pd.DataFrame()

    # Direct children of context_path
    context_path = get_submodule_by_path(model, 'backbone.context_path')
    if context_path is not None:
        context_children_tables[model_name] = child_breakdown(context_path, model_name, 'backbone.context_path')
    else:
        context_children_tables[model_name] = pd.DataFrame()

    # Direct children of spatial_path
    spatial_path = get_submodule_by_path(model, 'backbone.spatial_path')
    if spatial_path is not None:
        spatial_children_tables[model_name] = child_breakdown(spatial_path, model_name, 'backbone.spatial_path')
    else:
        spatial_children_tables[model_name] = pd.DataFrame()

    # Direct children of the fusion module
    fusion = get_submodule_by_path(model, 'backbone.ffm')
    if fusion is None:
        fusion = get_submodule_by_path(model, 'backbone.feature_fusion_module')
        fusion_parent = 'backbone.feature_fusion_module'
    else:
        fusion_parent = 'backbone.ffm'

    if fusion is not None:
        fusion_children_tables[model_name] = child_breakdown(fusion, model_name, fusion_parent)
    else:
        fusion_children_tables[model_name] = pd.DataFrame()

    # All named modules
    named_modules_dict = OrderedDict(model.named_modules())
    all_modules = all_named_modules_breakdown(model, model_name, root_label='model')
    all_modules_tables[model_name] = all_modules

    # Modules of interest (typical paths)
    interesting = find_interesting_modules(named_modules_dict)
    if not interesting.empty:
        interesting.insert(0, 'model_name', model_name)
    interesting_tables[model_name] = interesting

    # Attention / ARM / refinement-type modules
    attn_df = find_attention_like_modules(named_modules_dict)
    if not attn_df.empty:
        attn_df.insert(0, 'model_name', model_name)
    attention_tables[model_name] = attn_df

summary_df = pd.DataFrame(summary_rows).sort_values('total_params', ascending=False).reset_index(drop=True)
summary_df


In [ ]:

# =========================
# QUICK VIEW
# =========================

display(summary_df)

for model_name in MODELS_TO_CHECK:
    preview_df(f'{model_name} - hijos directos del modelo', root_children_tables[model_name], n=20)
    preview_df(f'{model_name} - hijos directos del backbone', backbone_children_tables[model_name], n=20)
    preview_df(f'{model_name} - hijos directos de backbone.context_path', context_children_tables[model_name], n=20)
    preview_df(f'{model_name} - hijos directos de backbone.spatial_path', spatial_children_tables[model_name], n=20)
    preview_df(f'{model_name} - hijos directos del fusion module', fusion_children_tables[model_name], n=20)
    preview_df(f'{model_name} - módulos de atención/ARM/refinement detectados', attention_tables[model_name], n=50)
    preview_df(f'{model_name} - módulos de interés (rutas típicas)', interesting_tables[model_name], n=50)


In [ ]:

# =========================
# COMPACT TABLE FOR THE PAPER
# =========================

paper_rows = []

for model_name in MODELS_TO_CHECK:
    model = models_cache[model_name]
    named_modules_dict = OrderedDict(model.named_modules())

    def add_row(label, path):
        mod = named_modules_dict.get(path, None)
        if mod is None:
            return
        total, trainable, non_trainable = count_params(mod)
        paper_rows.append({
            'model_name': model_name,
            'block_label': label,
            'path': path,
            'module_type': mod.__class__.__name__,
            'total_params': total,
            'trainable_params': trainable,
            'total_params_M': total / 1e6,
        })

    add_row('MODEL_TOTAL', '')
    add_row('BACKBONE', 'backbone')
    add_row('SPATIAL_PATH', 'backbone.spatial_path')
    add_row('CONTEXT_PATH', 'backbone.context_path')
    add_row('FUSION_MODULE_FFM', 'backbone.ffm')
    add_row('FUSION_MODULE_GENERIC', 'backbone.feature_fusion_module')
    add_row('ARMS_CONTAINER', 'backbone.context_path.arms')
    add_row('ATTN_REFINEMENT_CONTAINER', 'backbone.context_path.attention_refinement_modules')
    add_row('ARM16', 'backbone.context_path.arm16')
    add_row('ARM32', 'backbone.context_path.arm32')
    add_row('CONV_HEAD16', 'backbone.context_path.conv_head16')
    add_row('CONV_HEAD32', 'backbone.context_path.conv_head32')
    add_row('GAP_CONV', 'backbone.context_path.gap_conv')
    add_row('DECODE_HEAD', 'decode_head')
    add_row('CLS_HEAD', 'cls_head')

paper_df = pd.DataFrame(paper_rows)
paper_df
